# Intro to MLOps — churn example (notebook 2)

A second, even simpler hands-on: predict customer **churn** from a CSV with a
**Random Forest**, and let MLflow **autolog** do the tracking for you.

- Tutorial: <https://jienweng.github.io/notes/intro-to-mlops-tutorial/>

**The lazy way to track — autolog.** One line before you fit and MLflow captures
parameters, metrics, and the model automatically:

```python
import mlflow
mlflow.sklearn.autolog()   # tensorflow / pytorch / xgboost also supported
model.fit(...)             # params, metrics, and the model are logged for you
```

**How to use this notebook**
1. `pip install -r requirements.txt`
2. Run all cells. One MLflow run is logged — no manual `log_param/log_metric`.
3. Change `n_estimators` or `max_depth`, run again for a second run to compare.
4. `mlflow ui --backend-store-uri sqlite:///mlflow.db` → open <http://127.0.0.1:5000>.

## Setup

In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('churn-prediction')

# The lazy way: one line and every fit() is tracked automatically.
mlflow.sklearn.autolog()

/Users/jienweng/miniforge3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data — load the churn CSV

In [2]:
df = pd.read_csv('churn.csv')
print(f'Rows: {df.shape[0]}   Columns: {df.shape[1]}')
df.head()

Rows: 1000   Columns: 9


,tenure_months,monthly_charges,total_charges,contract,internet_service,tech_support,num_support_tickets,senior_citizen,churn
0,6,93.67,581.34,month-to-month,Fiber optic,Yes,9,1,No
1,56,108.64,6644.27,one-year,Fiber optic,No,1,0,No
2,47,112.11,5133.13,two-year,DSL,No,6,0,No
3,32,70.36,2217.78,two-year,Fiber optic,No,5,0,No
4,31,72.03,2372.32,month-to-month,Fiber optic,Yes,6,1,No


## Quick look at the data

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   tenure_months        1000 non-null   int64  
 1   monthly_charges      1000 non-null   float64
 2   total_charges        1000 non-null   float64
 3   contract             1000 non-null   object 
 4   internet_service     1000 non-null   object 
 5   tech_support         1000 non-null   object 
 6   num_support_tickets  1000 non-null   int64  
 7   senior_citizen       1000 non-null   int64  
 8   churn                1000 non-null   object 
dtypes: float64(2), int64(3), object(4)
memory usage: 70.4+ KB


In [4]:
# Columns, missing values, and how many customers churn
print('Missing values:', df.isna().sum().sum())
print()
print('Churn balance:')
print(df['churn'].value_counts())
print(f"Churn rate: {(df['churn'] == 'Yes').mean():.1%}")

Missing values: 0

Churn balance:
churn
No     675
Yes    325
Name: count, dtype: int64
Churn rate: 32.5%


In [5]:
# Numeric summary
df.describe().T

,count,mean,std,min,25%,50%,75%,max
tenure_months,1000.0,36.21500,21.070851,0.00,18.0000,36.00,55.0000,72.00
monthly_charges,1000.0,70.16252,29.393848,20.05,44.3675,69.35,96.2675,119.94
total_charges,1000.0,2533.96072,1924.355011,0.00,987.1925,2133.05,3692.0875,8904.31
num_support_tickets,1000.0,5.10100,3.105873,0.00,2.0000,5.00,8.0000,10.00
senior_citizen,1000.0,0.47200,0.499465,0.00,0.0000,0.00,1.0000,1.00


## Prepare features

Random Forest needs numbers, so one-hot encode the categorical columns and map
the target to 0/1. Then split into train and test.

In [6]:
y = (df['churn'] == 'Yes').astype(int)
X = pd.get_dummies(df.drop(columns='churn'), drop_first=True)
print(f'Feature columns after encoding: {X.shape[1]}')
print(list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

Feature columns after encoding: 10
['tenure_months', 'monthly_charges', 'total_charges', 'num_support_tickets', 'senior_citizen', 'contract_one-year', 'contract_two-year', 'internet_service_Fiber optic', 'internet_service_No', 'tech_support_Yes']


## Train with autolog

Change `n_estimators` / `max_depth` and re-run to produce another run to compare.
Autolog records the params, training metrics, and the model itself — we only add
two **test** metrics by hand.

In [7]:
n_estimators = 40
max_depth = 3

with mlflow.start_run(run_name=f'rf-n{n_estimators}-d{max_depth}'):
    model = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)   # autolog captures params, metrics, and the model

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    mlflow.log_metric('test_accuracy', acc)
    mlflow.log_metric('test_f1', f1)

    print(f'test accuracy={acc:.4f}, test f1={f1:.4f}')

2026/07/01 17:26:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/jienweng/miniforge3/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


2026/07/01 17:26:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/jienweng/miniforge3/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


2026/07/01 17:26:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


/Users/jienweng/miniforge3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


2026/07/01 17:26:51 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/jienweng/miniforge3/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


test accuracy=0.7200, test f1=0.4286


## Optional — grid search with autolog

Instead of guessing hyper-parameters, let `GridSearchCV` try every combination.
Autolog's default `max_tuning_runs=5` only logs child runs for the top 5
combinations by score — fine for a big search, but here we have just 6, so we
re-enable autolog with `max_tuning_runs=None` to log **every** combination as
its own child run, plus a parent run for the search itself.

In [8]:
from sklearn.model_selection import GridSearchCV

# Log every combination (not just the top 5) — there are only 6 here
mlflow.sklearn.autolog(max_tuning_runs=None)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 8, None],
}

with mlflow.start_run(run_name='rf-gridsearch'):
    search = GridSearchCV(
        RandomForestClassifier(random_state=42),
        param_grid, cv=3, scoring='f1', n_jobs=-1)
    search.fit(X_train, y_train)   # autolog logs the search + a child run per combo

    best = search.best_estimator_
    preds = best.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    mlflow.log_metric('test_accuracy', acc)
    mlflow.log_metric('test_f1', f1)

    print(f'best params: {search.best_params_}')
    print(f'test accuracy={acc:.4f}, test f1={f1:.4f}')

2026/07/01 17:26:51 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/jienweng/miniforge3/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


2026/07/01 17:26:54 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/jienweng/miniforge3/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


2026/07/01 17:26:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


/Users/jienweng/miniforge3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


2026/07/01 17:26:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/07/01 17:26:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/jienweng/miniforge3/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


best params: {'max_depth': None, 'n_estimators': 200}
test accuracy=0.7550, test f1=0.5664


## Compare runs

```shell
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Open <http://127.0.0.1:5000>, pick the `churn-prediction` experiment, select two
runs and click **Compare**. Notice how much autolog captured from a single
`fit()` — no manual logging of params or the model required.